# Regulatory RAG: Azure Databricks -> Foundry

Run this notebook **in Azure Databricks**, not a local Jupyter kernel. Provisioning, Unity Catalog permissions, serverless availability and the embedding endpoint must already be approved. The three-hour estimate is not a provisioning guarantee.

1. Run the dependency and setup cells. Upload the generated `data/pdfs/*.pdf` files and `data/manifest.json` to the printed managed Volume path (keep PDFs in a `pdfs` subdirectory).
2. Run ingestion/chunking. The default parser is `pypdf` for text PDFs; it does not perform OCR or reliably reconstruct complex tables. Empty/scanned documents are quarantined. `ai_parse_document` is an explicit optional branch requiring regional access and a supported runtime.
3. Set `confirm_paid_search` to `yes` only after cost approval, then provision/sync/query.
4. Copy the printed non-secret workspace/index settings into the local application's `.env`. Start with `RAG_MODE=databricks`; switch to `foundry` only after creating the new Foundry resource/project and saved agent.

The default catalog is `rag_demo`, deliberately not a production catalog. This notebook replaces only its dedicated demo tables. Public PDFs retain original source URLs; `tenant_id`, `acl_group`, classification and effective dates are **synthetic demo governance**, not claims about the source publication. No bearer tokens or secrets are printed. A successful filter test demonstrates a query predicate, not enterprise end-user authorization.

In [ ]:
%pip install pypdf==6.6.0 langchain-text-splitters==1.1.0 databricks-ai-search


In [ ]:
import re

for name, default in [('catalog', 'rag_demo'), ('schema', 'rag'), ('volume', 'raw_pdfs'), ('search_endpoint', 'regulatory-rag-search'), ('embedding_model', 'databricks-qwen3-embedding-0-6b')]:
    dbutils.widgets.text(name, default)
dbutils.widgets.dropdown('parser', 'pypdf', ['pypdf', 'ai_parse_document'])
dbutils.widgets.dropdown('confirm_paid_search', 'no', ['no', 'yes'])

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
volume = dbutils.widgets.get('volume')
for identifier in [catalog, schema, volume]:
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', identifier):
        raise ValueError('Catalog, schema and volume must be simple SQL identifiers.')
if catalog in {'risk_prod', 'prod', 'production'}:
    raise ValueError('Use a dedicated demo catalog, not a production catalog.')

namespace = f'{catalog}.{schema}'
volume_path = f'/Volumes/{catalog}/{schema}/{volume}'
bronze_table = f'{namespace}.bronze_docs'
silver_table = f'{namespace}.silver_docs'
gold_table = f'{namespace}.policy_chunks'
index_name = f'{namespace}.policy_chunks_index'
endpoint_name = dbutils.widgets.get('search_endpoint')
embedding_model = dbutils.widgets.get('embedding_model')

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {namespace}.{volume}')
print(f'Upload data/manifest.json to {volume_path}/manifest.json')
print(f'Upload data/pdfs/*.pdf to {volume_path}/pdfs/')
print('Search index:', index_name)
print('No search endpoint has been created by this cell.')

In [ ]:
import hashlib
import io
import json
from pathlib import Path
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

manifest = json.loads(Path(volume_path, 'manifest.json').read_text(encoding='utf-8'))
if not manifest or len({entry['doc_id'] for entry in manifest}) != len(manifest):
    raise ValueError('Manifest must be nonempty and contain unique document IDs.')
metadata = []
for entry in manifest:
    source = (Path(volume_path) / entry['path']).resolve()
    if not source.is_relative_to(Path(volume_path).resolve()) or source.suffix.lower() != '.pdf':
        raise ValueError('Manifest contains an unsafe or non-PDF source path.')
    if not 0 < source.stat().st_size <= 100 * 1024 * 1024:
        raise ValueError(f'Unsupported size: {source.name}')
    if hashlib.sha256(source.read_bytes()).hexdigest() != entry['sha256']:
        raise ValueError(f'Hash mismatch: {source.name}')
    if not entry.get('acl_group') or not entry.get('tenant_id'):
        raise ValueError('Missing demo authorization metadata.')
    metadata.append((source.name, *[str(entry.get(key, '')) for key in ['doc_id', 'title', 'source_url', 'document_version', 'effective_date', 'tenant_id', 'acl_group', 'classification', 'sha256']]))
fields = ['file_name', 'doc_id', 'title', 'source_url', 'document_version', 'effective_date', 'tenant_id', 'acl_group', 'classification', 'sha256']
metadata_df = spark.createDataFrame(metadata, T.StructType([T.StructField(name, T.StringType(), False) for name in fields]))
raw = (spark.read.format('binaryFile').load(f'{volume_path}/pdfs/*.pdf')
       .withColumn('file_name', F.regexp_extract('path', r'([^/]+)$', 1))
       .join(metadata_df, 'file_name', 'inner')
       .withColumn('ingested_at', F.current_timestamp()))
if raw.count() != len(manifest):
    raise ValueError('Uploaded PDFs do not match the manifest; check filenames and duplicate basenames.')
raw.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(bronze_table)

page_schema = T.ArrayType(T.StructType([
    T.StructField('page_number', T.IntegerType()),
    T.StructField('chunk_source', T.StringType()),
]))

def parse_pdf(content):
    try:
        reader = PdfReader(io.BytesIO(bytes(content)))
        if len(reader.pages) > 500:
            return []
        return [(number, page.extract_text() or '') for number, page in enumerate(reader.pages, 1)]
    except Exception:
        return []

if dbutils.widgets.get('parser') == 'pypdf':
    parsed = raw.withColumn('pages', F.udf(parse_pdf, page_schema)('content'))
    silver = (parsed.withColumn('page', F.explode('pages'))
              .select(*[F.col(name) for name in fields if name != 'file_name'], 'ingested_at', 'page.*'))
else:
    element_schema = T.StructType([
        T.StructField('document', T.StructType([
            T.StructField('elements', T.ArrayType(T.StructType([
                T.StructField('content', T.StringType()),
                T.StructField('type', T.StringType()),
                T.StructField('bbox', T.ArrayType(T.StructType([T.StructField('page_id', T.IntegerType())]))),
            ]))),
        ])),
    ])
    parsed = raw.withColumn('parsed', F.expr("ai_parse_document(content, map('version', '2.0', 'descriptionElementTypes', ''))"))
    structured = parsed.withColumn('payload', F.from_json(F.to_json('parsed'), element_schema))
    silver = (structured.withColumn('element', F.explode('payload.document.elements'))
              .filter(F.col('element.type').isin('text', 'table', 'section_header', 'title'))
              .select(*[F.col(name) for name in fields if name != 'file_name'], 'ingested_at',
                      (F.col('element.bbox')[0]['page_id'] + 1).alias('page_number'),
                      F.col('element.content').alias('chunk_source')))

silver = silver.filter((F.length(F.trim('chunk_source')) > 0) & F.col('page_number').isNotNull())
parsed_ids = silver.select('doc_id').distinct()
quarantine = raw.select('doc_id', 'file_name').join(parsed_ids, 'doc_id', 'left_anti').withColumn('parse_status', F.lit('no_readable_text_or_parse_failed'))
quarantine.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{namespace}.quarantine_docs')
silver = silver.withColumn('parse_status', F.lit('text_extracted')).withColumn('parser_version', F.lit(dbutils.widgets.get('parser')))
silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(silver_table)

# These settings count characters, not tokens.
def split_text(text):
    return RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80).split_text(text)

chunks = (silver.withColumn('parts', F.udf(split_text, T.ArrayType(T.StringType()))('chunk_source'))
          .select('*', F.posexplode('parts').alias('chunk_position', 'chunk_text'))
          .withColumn('chunk_id', F.sha2(F.concat_ws(':', 'doc_id', 'sha256', 'page_number', 'chunk_position', 'parser_version', F.lit('chars600-overlap80'), 'chunk_text'), 256))
          .select('chunk_id', 'doc_id', 'title', 'chunk_text', 'page_number', 'document_version', 'effective_date', 'tenant_id', 'acl_group', 'classification', 'source_url')
          .filter(F.length(F.trim('chunk_text')) > 0).dropDuplicates(['chunk_id']))
chunk_count = chunks.count()
if chunk_count == 0:
    raise ValueError('No readable chunks; inspect quarantine_docs or supply text PDFs.')
if not spark.catalog.tableExists(gold_table):
    chunks.write.format('delta').option('delta.enableChangeDataFeed', 'true').saveAsTable(gold_table)
else:
    (DeltaTable.forName(spark, gold_table).alias('target').merge(chunks.alias('source'), 'target.chunk_id = source.chunk_id')
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().whenNotMatchedBySourceDelete().execute())
spark.sql(f"ALTER TABLE {gold_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
assert spark.table(gold_table).count() == spark.table(gold_table).select('chunk_id').distinct().count()
print(json.dumps({'manifest_documents': len(manifest), 'parsed_documents': parsed_ids.count(), 'quarantined_documents': quarantine.count(), 'chunks': chunk_count}, indent=2))
display(spark.table(gold_table).select('doc_id', 'title', 'page_number', 'acl_group', 'effective_date').limit(12))

In [ ]:
if dbutils.widgets.get('confirm_paid_search') != 'yes':
    raise RuntimeError('Paid search creation is blocked. Approve pricing first, then set confirm_paid_search=yes.')

from databricks.ai_search.client import AISearchClient

search_client = AISearchClient()
endpoint_names = {item['name'] for item in search_client.list_endpoints().get('endpoints', [])}
if endpoint_name not in endpoint_names:
    search_client.create_endpoint(name=endpoint_name, endpoint_type='STANDARD')
endpoint_details = search_client.get_endpoint(name=endpoint_name)
if endpoint_details.get('endpoint_type', 'STANDARD') != 'STANDARD':
    raise ValueError('This notebook requires a STANDARD endpoint and dictionary filters.')
search_client.wait_for_endpoint(name=endpoint_name)

existing_indexes = {item['name'] for item in search_client.list_indexes(endpoint_name).get('vector_indexes', [])}
columns = ['doc_id', 'title', 'chunk_text', 'page_number', 'document_version', 'effective_date', 'tenant_id', 'acl_group', 'classification', 'source_url']
if index_name not in existing_indexes:
    search_index = search_client.create_delta_sync_index(
        endpoint_name=endpoint_name,
        source_table_name=gold_table,
        index_name=index_name,
        pipeline_type='TRIGGERED',
        primary_key='chunk_id',
        embedding_source_column='chunk_text',
        embedding_model_endpoint_name=embedding_model,
        columns_to_sync=columns,
    )
else:
    search_index = search_client.get_index(index_name=index_name)
    specification = search_index.describe().get('delta_sync_index_spec', {})
    if specification.get('source_table') != gold_table or specification.get('pipeline_type') != 'TRIGGERED':
        raise ValueError('Existing index does not match this demo table/triggered-sync configuration.')
    search_index.sync()
search_index.wait_until_ready(wait_for_updates=True)
print(json.dumps({'index': index_name, 'endpoint': endpoint_name, 'status': search_index.describe().get('status', {})}, indent=2, default=str))

In [ ]:
import time
from datetime import datetime, timezone

query = 'What approvals apply to elevated-risk commercial lending?'
filters = {'tenant_id': 'us-commercial', 'acl_group': ['compliance-analysts'], 'effective_date <=': '2026-11-09'}
result_columns = ['chunk_id', *columns]
started = time.perf_counter()
result = search_index.similarity_search(query_text=query, query_type='hybrid', filters=filters, columns=result_columns, num_results=6)
elapsed_ms = round((time.perf_counter() - started) * 1000)
print('Server-owned demo filters:', json.dumps(filters))
print('Actual Databricks response:', json.dumps(result, indent=2, default=str))

negative_filters = {**filters, 'acl_group': ['nonexistent-demo-cohort']}
negative = search_index.similarity_search(query_text=query, query_type='hybrid', filters=negative_filters, columns=result_columns, num_results=6)
assert not negative.get('result', {}).get('data_array', []), 'Denied cohort unexpectedly returned evidence.'

receipt = {
    'timestamp': datetime.now(timezone.utc).isoformat(), 'query': query,
    'filters': filters, 'elapsed_ms': elapsed_ms, 'index': index_name,
    'index_status': search_index.describe().get('status', {}),
    'documents': len(manifest), 'chunks': chunk_count,
    'response_schema': result.get('manifest', {}),
    'denied_cohort_returned_rows': 0,
    'scope': 'Actual Databricks search only; Foundry agent has not been called by this notebook.',
}
Path(volume_path, 'databricks-demo-receipt.json').write_text(json.dumps(receipt, indent=2, default=str), encoding='utf-8')
print(json.dumps(receipt, indent=2, default=str))
print('\nLocal application settings (not secrets):')
print('RAG_MODE=databricks')
print('DATABRICKS_HOST=https://' + spark.conf.get('spark.databricks.workspaceUrl').removeprefix('https://'))
print('DATABRICKS_INDEX=' + index_name)
print('\nFor Foundry: configure the NEW project endpoint, run python -m app.cloud, then set RAG_MODE=foundry.')
print('The Azure CLI principal needs workspace access, USE CATALOG, USE SCHEMA, SELECT on the index and embedding endpoint query permissions.')
print('Retain only approved demo resources. Idle indexes incur charges; endpoint billing may continue for 24 hours after last index deletion.')